In [1]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader


class UCIHARInertialDataset(Dataset):
    def __init__(self, root_dir: str, split: str = "train"):
        if split not in ("train", "test"):
            raise ValueError("split must be 'train' or 'test'")

        signal_names = (
            "total_acc_x", "total_acc_y", "total_acc_z",
            "body_acc_x", "body_acc_y", "body_acc_z",
            "body_gyro_x", "body_gyro_y", "body_gyro_z",
        )
        signal_dir = os.path.join(root_dir, split, "Inertial Signals")
        signals = [
            np.loadtxt(os.path.join(signal_dir, f"{name}_{split}.txt"))
            for name in signal_names
        ]
        x = np.stack(signals, axis=-1).astype(np.float32, copy=False)

        label_path = os.path.join(root_dir, split, f"y_{split}.txt")
        y = np.loadtxt(label_path, dtype=np.int64) - 1

        self.x = torch.from_numpy(x)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        return self.x[index], self.y[index]


In [2]:
root = "./Datasets/UCI HAR Dataset"

train_ds = UCIHARInertialDataset(root_dir=root, split="train")
test_ds = UCIHARInertialDataset(root_dir=root, split="test")

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False)

x, y = next(iter(train_loader))
print("batch shape:", x.shape)   # [32, 128, 9]  
print("label shape:", y.shape)   # [32]
print("label range:", y.min().item(), "~", y.max().item())


batch shape: torch.Size([32, 128, 9])
label shape: torch.Size([32])
label range: 0 ~ 5


In [3]:
len(train_ds), len(test_ds)

(7352, 2947)